# End-to-end runs of run_pipeline.py

**Code version:** `main` at 52835f3 (PR #11 included), merged into this branch.

**Test data:** one 87-minute raw recording from Audio Moth 5 (1,747 clips). The step-by-step notebooks (02 to 07) ran on the same clips, so each pipeline output can be compared with the notebook output.

**Prediction (from reading the code, before running):**
- Run 1, the pipeline as shipped: exit code 0 in about 2 minutes. Step 7 logs "Kept all 1,747 clips. AED labels: meaningful 1,747".
- Steps 2 to 6 outputs identical to the notebook outputs (those scripts did not change since the earlier runs).
- `features.csv`: 1,747 rows and 36 columns (29 features + 2 AED columns + 5 reference columns), identical to notebook 07's result.
- Run 2, with `--no_weather`: exit code 0, Step 6 skipped with no weather files, and 32 columns: run 1's 36 minus Temperature, Windspeed, Humidity and Volume_Wind_Ratio. All remaining columns identical to run 1, and the AED columns and the 25- and 23-feature lists are present.

**What this does:** sets up paths and prints the code version. It then checks that every input the pipeline reads exists, so a failure can't be blamed on a missing file.
**How to read it:** all `True` means any failure comes from the pipeline, not from missing inputs. The last line should say 0 old files in the run folders, so the results come only from these runs.

In [1]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
import subprocess, sys, shutil
from pathlib import Path
import pandas as pd

OUTPUTS_DIR = KWF_TEST_DIR / "outputs"          # outputs of the step-by-step notebooks
RAW_DIR     = KWF_TEST_DIR / "raw_audio"
RUN1        = KWF_TEST_DIR / "pipeline_run"
RUN2        = KWF_TEST_DIR / "pipeline_run_no_weather"
sys.path.insert(0, str(PIPELINE_DIR / "scripts"))

git = lambda *a: subprocess.run(["git", *a], cwd=PIPELINE_DIR, capture_output=True, text=True).stdout.strip()
print("Code version:", git("rev-parse", "--short", "HEAD"), "| branch:", git("branch", "--show-current"))

needed = [
    PIPELINE_DIR / "run_pipeline.py", PIPELINE_DIR / "config.py", PIPELINE_DIR / "requirements.txt",
    PIPELINE_DIR / "models" / "tinycnn_v3.pth", PIPELINE_DIR / "models" / "birdnet_fp16_pruned493.onnx",
    PIPELINE_DIR / "models" / "birdnet_fp16_pruned493.labels.txt", PIPELINE_DIR / "models" / "geomodel_allowlist_218.txt",
    RAW_DIR / "Audio_Moth_5_031925" / "20250319_163237.WAV",
    OUTPUTS_DIR / "nb07_features_asis.csv",
]
for p in needed:
    print(f"{p.exists()!s:5}  {p.name}")

for d in (RUN1, RUN2):
    shutil.rmtree(d, ignore_errors=True)
print("\nOld files in the run folders:", sum(len(list(d.rglob('*'))) for d in (RUN1, RUN2) if d.exists()))

Code version: 6c5be36 | branch: step1-4-testing-notebooks
True   run_pipeline.py
True   config.py
True   requirements.txt
True   tinycnn_v3.pth
True   birdnet_fp16_pruned493.onnx
True   birdnet_fp16_pruned493.labels.txt
True   geomodel_allowlist_218.txt
True   20250319_163237.WAV
True   nb07_features_asis.csv

Old files in the run folders: 0


**Result:** the code is main at 52835f3 (PR #11 included), merged into this branch as 6c5be36. All 9 inputs exist, and there were 0 old files in the run folders. The run folders were deleted before the runs.

## Run 1: run_pipeline.py as shipped

**What this does:** runs `python run_pipeline.py --raw_audio <raw> --output <run folder>` as a separate process, with no patches, and prints the key log lines and the files produced.
**How to read it:** exit code 0 means the pipeline finished. The step markers show each step starting, and the last one before an error is where it stopped. The log line "Kept all 1,747 clips. AED labels: ..." comes from Step 7. The file list should include `weather.csv` and `features.csv`. It takes about 2 minutes, and the output only appears at the end.

In [2]:
result1 = subprocess.run(
    [sys.executable, "run_pipeline.py", "--raw_audio", str(RAW_DIR), "--output", str(RUN1)],
    cwd=PIPELINE_DIR, capture_output=True, text=True,
)
lines = (result1.stdout + result1.stderr).splitlines()
print("Exit code:", result1.returncode)
print("Final line:", lines[-1] if lines else "(no output)")

print("\n--- step markers ---")
for l in lines:
    if any(k in l for k in ["STEP", "Weather   :", "Meaningful clips", "Kept all", "Done.", "Hourly weather", "Per-clip weather",
                            "WARNING", "ERROR", "Error", "Traceback"]):
        print(l)

print("\n--- files produced ---")
for p in sorted((RUN1 / "outputs").glob("*")):
    print(p.name)
print("clips written:", len(list((RUN1 / "clips").glob("*.wav"))))

Exit code: 0
Final line: 2026-11-18 14:42:59,974 — INFO — ============================================================

--- step markers ---
Done. 1,747 clips processed, 101 with detections.
Done. 1,747 rows saved, 0 failed.
Done. 1,747 rows saved to /Users/huyguy/kwf_test/pipeline_run/outputs/features.csv
2026-11-18 14:41:34,065 — INFO — Weather   : on
2026-11-18 14:41:34,066 — INFO — STEP 1 — Splitting raw audio into 3-second clips...
2026-11-18 14:41:37,358 — INFO — STEP 2 — Tiny CNN (AED): scoring every clip...
2026-11-18 14:41:45,672 — INFO — Meaningful clips: 1,747 / 1,747 total
2026-11-18 14:41:45,673 — INFO — STEP 3 — Compressed BirdNET: species + confidence...
2026-11-18 14:42:20,596 — INFO — STEP 4 — Acoustic feature extraction...
2026-11-18 14:42:46,888 — INFO — STEP 5 — Metadata extraction...
2026-11-18 14:42:58,906 — INFO — STEP 6 — Weather data fetch + matching...
2026-11-18 14:42:59,884 — INFO — Hourly weather saved to: /Users/huyguy/kwf_test/pipeline_run/outputs/weather

**Result:** exit code 0, about 85 s. All 8 steps ran, and Step 8 only logs the placeholder warning ("HP Model not yet available"). Step 7 logged "Kept all 1,747 clips. AED labels: meaningful 1,747". The files produced are acoustic_features.csv, birdnet_output.csv, features.csv, metadata.csv, tinycnn_predictions.csv, weather.csv and weather_hourly.csv, and 1,747 clips were written.

**What this does:** compares each file the pipeline wrote in run 1 with the file the matching step-by-step notebook made: same clips, same columns, largest numeric difference, same blanks, same text. Step 2's `audio_path` column is skipped, because its folder differs between runs.
**How to read it:** "same columns True", "same clips True" and "max numeric diff 0.0" mean the pipeline reproduced the notebook result exactly. Step 3 has no numeric columns, so only its text check counts. For `features.csv`, a difference around 1e-16 is floating-point rounding, and the check uses a tolerance of 1e-12.

In [3]:
import numpy as np
O1 = RUN1 / "outputs"

def compare(label, nb_file, run_file, drop=()):
    a = pd.read_csv(OUTPUTS_DIR / nb_file).drop(columns=list(drop)).sort_values("clip_name").reset_index(drop=True)
    b = pd.read_csv(O1 / run_file).drop(columns=list(drop)).sort_values("clip_name").reset_index(drop=True)
    same_cols = a.columns.tolist() == b.columns.tolist()
    same_clips = same_cols and a["clip_name"].equals(b["clip_name"])
    print(f"\n{label}: {a.shape} vs {b.shape} | same columns {same_cols} | same clips {same_clips}")
    if not same_clips:
        return
    num = a.select_dtypes("number").columns
    txt = [c for c in a.columns if c not in num]
    diff = float((a[num] - b[num]).abs().max().max()) if len(num) else "no numeric columns"
    print("   max numeric diff:", diff,
          "| blanks same:", a.isna().equals(b.isna()),
          "| text same:", all(a[c].equals(b[c]) for c in txt))

compare("Step 2 Tiny CNN", "nb02_recorder5_rerun_predictions.csv", "tinycnn_predictions.csv", drop=["audio_path"])
compare("Step 3 BirdNET", "nb03_birdnet_output.csv", "birdnet_output.csv")
compare("Step 4 acoustic", "nb04_recorder5_acoustic_features.csv", "acoustic_features.csv")
compare("Step 5 metadata", "nb05_metadata.csv", "metadata.csv")
compare("Step 6 weather", "nb06_weather_matches.csv", "weather.csv")
compare("Step 7 features", "nb07_features_asis.csv", "features.csv")

f1 = pd.read_csv(O1 / "features.csv")
print("\nStep 7 largest numeric difference is within 1e-12:",
      bool(np.allclose(f1.select_dtypes("number").sort_index(axis=1),
                       pd.read_csv(OUTPUTS_DIR / "nb07_features_asis.csv").select_dtypes("number").sort_index(axis=1), rtol=0, atol=1e-12)))
print("features.csv:", f1.shape, "| blanks in weather features:",
      int(f1[["Temperature", "Windspeed", "Humidity", "Volume_Wind_Ratio"]].isna().sum().sum()))
print("AED labels:", f1["predicted_label"].value_counts().to_dict())


Step 2 Tiny CNN: (1747, 5) vs (1747, 5) | same columns True | same clips True
   max numeric diff: 0.0 | blanks same: True | text same: True

Step 3 BirdNET: (1747, 3) vs (1747, 3) | same columns True | same clips True
   max numeric diff: no numeric columns | blanks same: True | text same: True

Step 4 acoustic: (1747, 11) vs (1747, 11) | same columns True | same clips True
   max numeric diff: 0.0 | blanks same: True | text same: True

Step 5 metadata: (1747, 12) vs (1747, 12) | same columns True | same clips True
   max numeric diff: 0.0 | blanks same: True | text same: True

Step 6 weather: (1747, 12) vs (1747, 12) | same columns True | same clips True
   max numeric diff: 0.0 | blanks same: True | text same: True

Step 7 features: (1747, 36) vs (1747, 36) | same columns True | same clips True
   max numeric diff: 0.0 | blanks same: True | text same: True

Step 7 largest numeric difference is within 1e-12: True
features.csv: (1747, 36) | blanks in weather features: 0
AED labels: {

**Result:** the outputs of Steps 2, 3, 4, 5, 6 and 7 are identical to the step-by-step notebook outputs: same clips, same columns, same blanks and text, and a maximum numeric difference of 0.0 (Step 3 has no numeric columns). `features.csv` has 1,747 rows and 36 columns, with 0 blank weather values. AED labels: meaningful 1,747. The pipeline passes the same files from step to step as the individual notebooks do. Both use the same code, so this shows consistency; the comparison with the code before PR #11 is in notebook 07.

## Run 2: run_pipeline.py --no_weather

**What this does:** runs the same command with `--no_weather`, then compares its `features.csv` with run 1's and checks it against the 25- and 23-feature lists.

**How to read it:** "Skipped" for Step 6 and no weather files confirm the flag works. The columns missing compared with run 1 should be exactly the four weather-based ones, leaving 32 columns. "Remaining columns identical" means the flag changed nothing else. Both lists should show "missing: none".

In [4]:
from hp_features import HP_FEATURES_25, HP_FEATURES_23

result2 = subprocess.run(
    [sys.executable, "run_pipeline.py", "--raw_audio", str(RAW_DIR), "--output", str(RUN2), "--no_weather"],
    cwd=PIPELINE_DIR, capture_output=True, text=True,
)
lines = (result2.stdout + result2.stderr).splitlines()
print("Exit code:", result2.returncode, "| final line:", lines[-1] if lines else "")
print([l for l in lines if "STEP 6" in l or "Weather   :" in l or "Kept all" in l])
print("Files:", sorted(p.name for p in (RUN2 / "outputs").glob("*")))

f2 = pd.read_csv(RUN2 / "outputs" / "features.csv").sort_values("clip_name").reset_index(drop=True)
f1s = f1.sort_values("clip_name").reset_index(drop=True)
print("\nShape:", f2.shape, "| columns missing vs run 1:", [c for c in f1s.columns if c not in f2.columns])
same = f1s[f2.columns]
num = f2.select_dtypes("number").columns
print("Remaining columns identical to run 1:",
      bool(np.allclose(same[num], f2[num], rtol=0, atol=1e-12)) and all(same[c].equals(f2[c]) for c in f2.columns if c not in num))
print("HP_FEATURES_25 missing:", [c for c in HP_FEATURES_25 if c not in f2.columns] or "none",
      "| HP_FEATURES_23 missing:", [c for c in HP_FEATURES_23 if c not in f2.columns] or "none")
print("AED columns present:", all(c in f2.columns for c in ["not_meaningful_prob", "predicted_label"]))

Exit code: 0 | final line: 2026-11-18 14:44:26,044 — INFO — ============================================================
['2026-11-18 14:43:01,716 — INFO — Weather   : off (--no_weather)', '2026-11-18 14:44:25,977 — INFO — STEP 6 — Skipped (--no_weather)', '2026-11-18 14:44:26,044 — INFO — Kept all 1,747 clips. AED labels: meaningful 1,747']
Files: ['acoustic_features.csv', 'birdnet_output.csv', 'features.csv', 'metadata.csv', 'tinycnn_predictions.csv']

Shape: (1747, 32) | columns missing vs run 1: ['Temperature', 'Windspeed', 'Humidity', 'Volume_Wind_Ratio']
Remaining columns identical to run 1: True
HP_FEATURES_25 missing: none | HP_FEATURES_23 missing: none
AED columns present: True


**Result:** exit code 0, about 85 s. The log shows "Weather: off (--no_weather)" and "STEP 6 — Skipped (--no_weather)", and no weather files were written. `features.csv` has 1,747 rows and 32 columns: exactly four columns are absent compared with run 1 (Temperature, Windspeed, Humidity, Volume_Wind_Ratio), and the remaining columns are identical to run 1. Every name in HP_FEATURES_25 and HP_FEATURES_23 is present, and the AED columns are kept.

## Historical runs (earlier code)

- **Code 4f86604 (before PR #2):** the same command stopped at Step 6 with `FileNotFoundError: 'path/to/raw'` (exit code 1). Step 6 read placeholder paths from `config.py` and ignored `--raw_audio`, so no `weather.csv` or `features.csv` was written.
- **Code 4f86604 with five patches in a scratch copy:** exit code 0 in about 110 s. The patches filled in the paths and renamed the weather columns, and are no longer needed.
- **Code 4104a17 (after PR #2):** exit code 0 in about 114 s. Weather had 3 hourly rows and 0 blanks (the old code left 599 clips blank), and `features.csv` had 34 columns. `--no_weather` gave 30.
- **Main at 52835f3 (PR #11):** Step 7 keeps every clip and adds the two AED columns, which is why the column counts are now 36 and 32.

## Observations (main at 52835f3)

**Tested:** `run_pipeline.py` end to end on one 87-minute recording (Audio Moth 5, 1,747 clips), with and without `--no_weather`.

**Result:** both runs finish with exit code 0, in about 85 s each. With weather, `features.csv` has 1,747 rows and 36 columns (29 features, the AED columns `not_meaningful_prob` and `predicted_label`, and 5 reference columns), with 0 blank weather values. With `--no_weather` it has 32 columns, with exactly the four weather-based columns absent. The outputs of Steps 2 to 7 match the step-by-step notebooks (maximum numeric difference 0.0), so the pipeline passes the same files between steps as the individual runs.

**Not tested:** the HP model (Step 8 is a placeholder); more than one recorder or recording in a run; real clips flagged `not_meaningful` (recorder 5 has none); a run in a clean environment built from `requirements.txt` (these runs used my environment, with different package versions); a failed weather download.